# Description

It shows the pathways enriched (from the MultiPLIER models) given an LV name (in Settings below).
These "pathways enriched" are a set of limited pathways used during training of this PLIER model.
More pathways might be enriched if using external and more comprehensive databases such as gProfiler or FUMA as shown below.

# Modules loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
from pathlib import Path

import pandas as pd

from entity import Trait
import conf

# Settings

In [3]:
LV_NAME = "LV913"

# Paths

In [4]:
OUTPUT_FIGURES_DIR = Path(conf.RESULTS_DIR, "demo", f"{LV_NAME.lower()}").resolve()
display(OUTPUT_FIGURES_DIR)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

PosixPath('/opt/data/results/demo/lv913')

# Load MultiPLIER summary

In [5]:
multiplier_model_summary = pd.read_pickle(conf.MULTIPLIER["MODEL_SUMMARY_FILE"])

In [6]:
multiplier_model_summary.shape

(2157, 5)

In [7]:
multiplier_model_summary.head()

,pathway,LV index,AUC,p-value,FDR
1,KEGG_LYSINE_DEGRADATION,1,0.388059,0.866078,0.956005
2,REACTOME_MRNA_SPLICING,1,0.733057,0.000048,0.000582
3,MIPS_NOP56P_ASSOCIATED_PRE_RRNA_COMPLEX,1,0.680555,0.001628,0.011366
4,KEGG_DNA_REPLICATION,1,0.549473,0.312155,0.539951
5,PID_MYC_ACTIVPATHWAY,1,0.639303,0.021702,0.083739


# LV pathways

In [8]:
lv_pathways = multiplier_model_summary[
    multiplier_model_summary["LV index"].isin((LV_NAME[2:],))
    & (
        (multiplier_model_summary["FDR"] < 0.05)
                | (multiplier_model_summary["AUC"] >= 0.75)
    )
]

In [9]:
lv_pathways.shape

(1, 5)

In [10]:
lv_pathways = lv_pathways[["pathway", "AUC", "FDR"]].sort_values("FDR")

In [11]:
lv_pathways = lv_pathways.assign(AUC=lv_pathways["AUC"].apply(lambda x: f"{x:.2f}"))

In [12]:
lv_pathways = lv_pathways.assign(FDR=lv_pathways["FDR"].apply(lambda x: f"{x:.2e}"))

In [13]:
lv_pathways = lv_pathways.rename(
    columns={
        "pathway": "Pathway",
    }
)

In [14]:
lv_pathways.head()

,Pathway,AUC,FDR
2016,KEGG_COMPLEMENT_AND_COAGULATION_CASCADES,0.75,1.58e-02


# Load LV data

In [15]:
from data.recount2 import LVAnalysis

In [16]:
lv_obj = LVAnalysis(LV_NAME)

Here I show the top 20 genes for our LV. You can see gene symbols, the LV weight (in column `LV603`) and the cytoband.

In [17]:
lv_obj.lv_genes.head(20)

,gene_name,LV913,gene_band
0,PLA2G2D,5.509843,1p36.12
1,IGKC,4.995934,NaN
2,CCL19,3.842465,9p13.3
3,DERL3,3.381121,22q11.23
4,NOS2,3.315122,17q11.2
5,CXCL13,3.243843,4q21.1
6,OLFM4,2.765351,13q14.3
7,C1QB,2.169730,1p36.12
8,MZB1,2.124240,5q31.2
9,IGHM,2.093565,NaN


# Pathway enrichment using external databases

## gProfiler

In [18]:
print(" ".join(lv_obj.lv_genes.head(70)["gene_name"].tolist()))

PLA2G2D IGKC CCL19 DERL3 NOS2 CXCL13 OLFM4 C1QB MZB1 IGHM MADCAM1 CR2 TNFRSF6B ADAMDEC1 C2 C1QC CASP5 GMDS TMEM176B C1S C1QA TMEM176A GABBR1 TAP1 UGT2B17 PIGR CD74 ACSL5 IGLL1 RARRES3 MEOX1 RAMP3 KCNE3 LCN2 GGT5 SLC40A1 GUCY2C CFB VPREB3 TFF3 CASP10 XBP1 ANPEP DNASE1L3 SERPING1 CETP CLDN15 TAP2 COL15A1 CPA3 GBP3 STAT1 PSMB8 MMP12 SLAMF8 NR1I2 ETS2 CCL18 CLDN7 KCNQ1 FCGBP RELB PLCB3 CTSH PDZK1IP1 CXCL9 CASP1 LAP3 LGALS3BP MYO7B


Copy/paste the list of genes above and use gProfiler: https://biit.cs.ut.ee/gprofiler/gost

Results URL: https://biit.cs.ut.ee/gplink/l/alcx61S2mT5

**Notes**: seems more immune system related.

## FUMA

In [19]:
# print top genes in module
print("\n".join(lv_obj.lv_genes.head(70)["gene_name"].tolist()))

PLA2G2D
IGKC
CCL19
DERL3
NOS2
CXCL13
OLFM4
C1QB
MZB1
IGHM
MADCAM1
CR2
TNFRSF6B
ADAMDEC1
C2
C1QC
CASP5
GMDS
TMEM176B
C1S
C1QA
TMEM176A
GABBR1
TAP1
UGT2B17
PIGR
CD74
ACSL5
IGLL1
RARRES3
MEOX1
RAMP3
KCNE3
LCN2
GGT5
SLC40A1
GUCY2C
CFB
VPREB3
TFF3
CASP10
XBP1
ANPEP
DNASE1L3
SERPING1
CETP
CLDN15
TAP2
COL15A1
CPA3
GBP3
STAT1
PSMB8
MMP12
SLAMF8
NR1I2
ETS2
CCL18
CLDN7
KCNQ1
FCGBP
RELB
PLCB3
CTSH
PDZK1IP1
CXCL9
CASP1
LAP3
LGALS3BP
MYO7B


In [20]:
# save all genes in model to use as background list of genes
lv_obj.lv_genes["gene_name"].to_csv(OUTPUT_FIGURES_DIR / "all_genes.txt", header=None, index=False)

In [21]:
OUTPUT_FIGURES_DIR / "all_genes.txt"

PosixPath('/opt/data/results/demo/lv913/all_genes.txt')

In [22]:
!head /opt/data/results/demo/lv24/all_genes.txt

KCNK7
KRT1
ACER1
ASPRV1
CDHR1
CTNNBIP1
CAPNS2
ELOVL3
KLC3
DSP


In [23]:
!wc -l /opt/data/results/demo/lv24/all_genes.txt

6750 /opt/data/results/demo/lv24/all_genes.txt


Now go to the FUMA GENE2FUNC module here: https://fuma.ctglab.nl/gene2func

1. Paste the list of top genes above and then upload the `all_genes.txt` file.
2. Use a "Title" and click on "Submit"

**Notes:** significantly expressed in small intestine, colon, spleen, kidney